In [ ]:
import duckdb

con = duckdb.connect("ipeds.duckdb")

query = """
SELECT 
    s.unitid,
    s.year,
    h.institution_name,
    h.city,
    h.state,
    s.arank,
    CASE 
        WHEN s.arank = 1 THEN 'Full'
        WHEN s.arank = 2 THEN 'Associate'
        WHEN s.arank = 3 THEN 'Assistant'
    END AS rank,
    s.saeq9at,
    s.satotlt,
    s.saoutlt,
    s.sa09mct, s.sa10mct, s.sa11mct, s.sa12mct,
    s.sa09mat, s.sa10mat, s.sa11mat, s.sa12mat

FROM sal_is s
LEFT JOIN hd h 
    ON s.unitid = h.unitid 
   AND s.year = h.year

WHERE s.arank IN (1, 2, 3)
"""

df = con.sql(query).df()

print(df.shape)
df.head()



In [ ]:


# UPG UNITIDs + Siena 195474
upg_unitids = [195474,
               215929, 216278, 167996, 218070, 163046, 123554,
               203368, 209825, 213251, 110413, 214175, 214157,
               209056, 152390, 150163, 217536, 184348, 136950,
               164562, 193584, 221519, 165699, 213507, 195216,
               212197, 215770, 236230, 183239, 239716, 146481,
               134079, 237066, 224323, 191968, 221351, 230816]

# 6. Create UPG

upg = df[df['unitid'].isin(upg_unitids)].copy()

print('Dataframe \t Rows \t Columns')
print('df         \t', df.shape[0], '\t', df.shape[1])
print('upg        \t', upg.shape[0], '\t', upg.shape[1])

upg



In [ ]:
sa = upg.fillna(0).copy()
sa['avgsal'] = (sa['sa09mat']*sa['sa09mct'] + 
                 sa['sa10mat']*sa['sa10mct'] + 
                 sa['sa11mat']*sa['sa11mct'] + 
                 sa['sa12mat']*sa['sa12mct']) / (sa['sa09mct'] + sa['sa10mct'] + sa['sa11mct'] + sa['sa12mct'])
sa['argave'] = sa['saoutlt'] / sa['satotlt']
sa

In [ ]:
full_avg = (
    sa[sa['rank'] == 'Full']
    .groupby(['unitid'], as_index=False)['argave']
    .mean()
    .rename(columns={'argave': 'average_salary'})
)

# Add institution name from year 2024
names_2024 = (
    sa[['unitid', 'institution_name','year']]
    .query('year == 2024')
    .drop_duplicates(subset=['unitid'])          # in case of any duplicates
)

full_avg = full_avg.merge(
    names_2024, 
    on='unitid', 
    how='left'
)

# Final sorting
full_avg = (
    full_avg
    .sort_values('average_salary', ascending=False)
    .reset_index(drop=True)    
)

full_avg